In [ ]:
# Regression with sklearn - diabetes
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import root_mean_squared_error, r2_score

df = pd.read_csv("../data/diabetes.csv")
X = df.drop(columns="disease_progression").values
y = df["disease_progression"].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0)

In [ ]:
# 2.1 models (scaler + model in one pipeline)
models = {"Linear": LinearRegression(), "Ridge": Ridge(alpha=10), "KNN": KNeighborsRegressor(10),
          "RandomForest": RandomForestRegressor(random_state=0), "SVR": SVR(C=100, gamma=0.01)}
models = {k: make_pipeline(StandardScaler(), m).fit(Xtr, ytr) for k, m in models.items()}

In [ ]:
# 2.1 benchmark / ranking (lower RMSE = better)
res = pd.DataFrame({k: [root_mean_squared_error(yte, m.predict(Xte)), r2_score(yte, m.predict(Xte)),
                        cross_val_score(m, X, y, cv=5).mean()] for k, m in models.items()},
                   index=["RMSE", "R2", "CV_R2"]).T.sort_values("RMSE")
res

In [ ]:
# 2.2 predict a new sample with the best model (order: age sex bmi bp s1 s2 s3 s4 s5 s6)
best = res.index[0]
print(best, models[best].predict([[50, 2, 28.0, 95, 200, 120, 50, 4.0, 4.8, 90]])[0].round(1))